# Chapter 04: Bird's-Eye View (BEV) Transform (Lift, Splat, Shoot)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/04_bev_lift_splat_shoot.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do we transform 8 perspective cameras into a metric 3D top-down grid without LiDAR?*

---

## 1. 🚨 The Real-World Dilemma
Perspective cameras break Euclidean geometry. Lift-Splat-Shoot (Philion & Fidler ECCV 2020) predicts categorical depth distributions along camera rays, projects features into 3D space, and pools them into metric BEV voxels.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

class ToyLiftSplat(nn.Module):
    def __init__(self, D=20, d_min=2.0, d_max=40.0, C=16):
        super().__init__()
        self.D = D
        self.C = C
        self.depth_bins = torch.linspace(d_min, d_max, D)
        self.conv = nn.Conv2d(32, D + C, kernel_size=1)

    def forward(self, x):
        B, _, H, W = x.shape
        logits = self.conv(x)
        depth_logits = logits[:, :self.D]
        context = logits[:, self.D:]
        depth_prob = F.softmax(depth_logits, dim=1)
        frustum = depth_prob.unsqueeze(2) * context.unsqueeze(1)
        return frustum, depth_prob

model = ToyLiftSplat()
dummy_features = torch.randn(1, 32, 16, 16)
frustum, depth_prob = model(dummy_features)

print(f"Frustum tensor: {frustum.shape} (Batch, Depth, Channels, Height, Width)")
print(f"Depth prob sum along ray: {depth_prob[0, :, 8, 8].sum().item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(model.depth_bins.numpy(), depth_prob[0, :, 8, 8].detach().numpy(), 'o-', color="#58a6ff", lw=2)
plt.title("Categorical Depth Probability Along Camera Sightline")
plt.xlabel("Depth (m)")
plt.ylabel("Probability")
plt.grid(True, alpha=0.3)
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Obstacles smeared radially along camera sightlines** | Depth distribution entropy too high (uniform guessing). | Compute $-\sum p_i \log p_i$. If $>3.0$, model has zero depth confidence. | Add auxiliary depth supervision with sparse LiDAR or radar. |
| **All obstacles shift sideways during acceleration** | Pitch/squat dynamic chassis tilt rotates $R_{\text{ext}}$. | Correlate position error with chassis pitch IMU. | Feed live IMU suspension pitch into extrinsic rotation matrix. |